In [1]:
!pip install kafka-python beautifulsoup4 requests pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 326.1/326.1 kB 11.5 MB/s eta 0:00:00


In [5]:
import requests
from bs4 import BeautifulSoup
from kafka import KafkaProducer
import json
import time
import re

print("INICIANDO SCRAPER MASIVO DE REVIEWS (Objetivo: 500 juegos x 100 reseñas)...")

# --- CONFIGURACIÓN KAFKA ---
try:
    producer = KafkaProducer(
        bootstrap_servers=['kafka:9092'],
        value_serializer=lambda v: json.dumps(v).encode('utf-8')
    )
    print("   Conectado a Kafka.")
except Exception as e:
    print(f"   Error de conexión Kafka: {e}")
    producer = None

# --- FUNCIÓN PARA EXTRAER RESEÑAS DE LA API ---
def obtener_reviews(appid, titulo):
    url_api = f"https://store.steampowered.com/appreviews/{appid}?json=1&filter=recent&language=all&num_per_page=100"
    
    try:
        res = requests.get(url_api, timeout=10)
        data = res.json()
        
        reviews_enviadas = 0
        if 'reviews' in data:
            for r in data['reviews']:
                review_data = {
                    "review_id": r['recommendationid'], # UPSERT KEY
                    "appid": appid,
                    "titulo_juego": titulo,
                    "review": r['review'],
                    "voted_up": r['voted_up'],
                    "language": r['language'],
                    "playtime_hours": round(r['author'].get('playtime_forever', 0) / 60, 1) if 'author' in r else 0,
                    "timestamp": time.strftime('%Y-%m-%d %H:%M:%S')
                }
                producer.send('steam_reviews', value=review_data)
                reviews_enviadas += 1
        return reviews_enviadas
    except Exception as e:
        print(f"      ⚠️ Error en API para {titulo}: {e}")
        return 0

# --- FUNCIÓN PRINCIPAL DE SCRAPING CON PAGINACIÓN ---
def ejecutar_scraping_completo():
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    
    juegos_objetivo = 500
    juegos_procesados = 0
    pagina = 1
    total_reviews = 0
    
    print(f"Buscando los {juegos_objetivo} juegos Indie más vendidos...")

    try:
        # Bucle que pasará de página en página hasta llegar a 500
        while juegos_procesados < juegos_objetivo:
            # Añadimos &page= al final de la URL
            url_busqueda = f"https://store.steampowered.com/search/?filter=topsellers&tags=492&page={pagina}"
            response = requests.get(url_busqueda, headers=headers)
            soup = BeautifulSoup(response.text, 'html.parser')
            rows = soup.select('.search_result_row')
            
            # Si una página no tiene juegos, hemos llegado al final de Steam
            if not rows:
                print("No hay más juegos en la lista de Steam. Terminando búsqueda.")
                break
                
            print(f"\n--- Escaneando Página {pagina} de Steam ---")

            for item in rows:
                # Si ya llegamos a 500, cortamos el bucle
                if juegos_procesados >= juegos_objetivo:
                    break
                    
                titulo = item.select_one('.title').text.strip()
                url_juego = item['href']
                
                # Extraer AppID con Regex
                match = re.search(r'/app/(\d+)/', url_juego)
                if match:
                    appid = match.group(1)
                    num_revs = obtener_reviews(appid, titulo)
                    total_reviews += num_revs
                    juegos_procesados += 1
                    
                    print(f"   [{juegos_procesados}/{juegos_objetivo}] -> {titulo} (AppID: {appid}): {num_revs} reseñas enviadas.")
                    
                    # Pausa técnica FUNDAMENTAL (sin esto, Steam bloquea tu IP de AWS)
                    time.sleep(0.5)
            
            # Pasamos a la siguiente página web
            pagina += 1
            time.sleep(2) # Pausa larga entre páginas web completas
            
        producer.flush()
        print(f"\nPROCESO FINALIZADO.")
        print(f"Total de juegos escaneados: {juegos_procesados}")
        print(f"Total de reseñas inyectadas en Kafka: {total_reviews}")

    except Exception as e:
        print(f"Error crítico en el scraping: {e}")

# --- EJECUCIÓN ---
if producer:
    ejecutar_scraping_completo()
else:
    print("No se pudo iniciar el script porque Kafka no está disponible.")

INICIANDO SCRAPER MASIVO DE REVIEWS (Objetivo: 500 juegos x 100 reseñas)...
   Conectado a Kafka.
Buscando los 500 juegos Indie más vendidos...

--- Escaneando Página 1 de Steam ---
   [1/500] -> MOUSE: P.I. For Hire (AppID: 2416450): 100 reseñas enviadas.
   [2/500] -> IdleOn (AppID: 1476970): 0 reseñas enviadas.
   [3/500] -> Slay the Spire 2 (AppID: 2868840): 98 reseñas enviadas.
   [4/500] -> Rust (AppID: 252490): 100 reseñas enviadas.
   [5/500] -> Kerbal Space Program (AppID: 220200): 100 reseñas enviadas.
   [6/500] -> No Man's Sky (AppID: 275850): 100 reseñas enviadas.
   [7/500] -> RimWorld (AppID: 294100): 100 reseñas enviadas.
   [8/500] -> REPLACED (AppID: 1663850): 100 reseñas enviadas.
   [9/500] -> Risk of Rain 2 (AppID: 632360): 100 reseñas enviadas.
   [10/500] -> Retro Rewind - Video Store Simulator (AppID: 3552140): 99 reseñas enviadas.
   [11/500] -> Hades II (AppID: 1145350): 99 reseñas enviadas.
   [12/500] -> Travellers Rest (AppID: 1139980): 100 reseñas enviadas